In [ ]:
!pip install pandas_ta yfinance -q

In [ ]:
# memory.py
USER_MEMORY = {
    "default_user": {
        "risk_profile": "moderate",
        "history": []
    }
}

def get_user_profile(user_id="default_user"):
    return USER_MEMORY.get(user_id, USER_MEMORY["default_user"])

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

SecretNotFoundError: Secret OPENAI_API_KEY does not exist.

In [ ]:
class State(TypedDict):
    # input
    query: str
    ticker: str

    # ⭐ NEW: user memory
    user_profile: Dict[str, Any]

    # Person 1
    market_data: Dict[str, Any]
    technical_signals: Dict[str, Any]
    options_data: Dict[str, Any]

    price_data: Any
    technical_full: Dict[str, Any]
    options_full: Dict[str, Any]

    # Person 2
    news_sentiment: Dict[str, Any]

    # Person 3
    risk_score: float
    weighted_signals: Dict[str, Any]

    # ⭐ NEW: LLM reasoning
    llm_analysis: str

    # output
    decision: Dict[str, Any]

NameError: name 'TypedDict' is not defined

Christine:

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')
def get_market_data(ticker: str, period: str = "3mo") -> dict:
  """
    Fetch historical price data and compute basic statistics.

    Parameters:
        ticker: Stock symbol, e.g. "NVDA"
        period: Data time range, default "3mo" (3 months)

    Returns:
        dict with keys:
            - ticker: str
            - price_data: pd.DataFrame (OHLCV + derived columns)
  """
  # download data
  try:
        df = yf.download(ticker, period=period, progress=False)
  except Exception as e:
        return {"error": f"Failed to download data for {ticker}: {str(e)}"}

  if df.empty:
        return {"error": f"No data returned for {ticker}"}

  if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.droplevel(1)
  df['Daily_Return'] = df['Close'].pct_change()
  df['Cumulative_Return'] = (1 + df['Daily_Return']).cumprod() - 1
  df['Rolling_Volatility'] = df['Daily_Return'].rolling(window=20).std() * np.sqrt(252)
  df['Rolling_Max'] = df['Close'].cummax()
  df['Drawdown'] = (df['Close'] - df['Rolling_Max']) / df['Rolling_Max']

  returns_clean = df['Daily_Return'].dropna()
  current_price = float(df['Close'].iloc[-1])
  daily_vol = float(returns_clean.std())
  stats = {
        "current_price": current_price,
        "period_high": float(df['High'].max()),
        "period_low": float(df['Low'].min()),
        "total_return": float(df['Cumulative_Return'].iloc[-1]),
        "avg_daily_return": float(returns_clean.mean()),
        "daily_volatility": daily_vol,
        "annualized_volatility": float(daily_vol * np.sqrt(252)),
        "max_drawdown": float(df['Drawdown'].min()),
        "sharpe_ratio": float((returns_clean.mean() / daily_vol) * np.sqrt(252)) if daily_vol > 0 else 0.0,
        "sortino_ratio": _calculate_sortino(returns_clean),
        "var_95": float(returns_clean.quantile(0.05)),
        "cvar_95": float(returns_clean[returns_clean <= returns_clean.quantile(0.05)].mean()) if len(returns_clean[returns_clean <= returns_clean.quantile(0.05)]) > 0 else 0.0,  # 条件VaR
        "skewness": float(returns_clean.skew()),
        "kurtosis": float(returns_clean.kurtosis()),
        "avg_daily_volume": float(df['Volume'].mean()),
        "recent_5d_return": float(df['Close'].iloc[-1] / df['Close'].iloc[-6] - 1) if len(df) >= 6 else 0.0,
        "recent_20d_return": float(df['Close'].iloc[-1] / df['Close'].iloc[-21] - 1) if len(df) >= 21 else 0.0,
        "trading_days": len(df),
        "date_range": f"{df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')}",
    }
  return {
      "ticker": ticker,
      "price_data": df,
      "stats": stats,
    }




In [ ]:
# technical analysis agent
def get_technical_indicators(df: pd.DataFrame) -> dict:
    """
    Compute a full suite of technical indicators and generate rule-based signals.

    Parameters:
        df: DataFrame with OHLCV columns (from get_market_data price_data)

    Returns:
        dict with keys:
            - indicators: dict of latest indicator values
            - signals: dict of rule-based signal interpretations
            - signal_summary: str overview of bullish/bearish count
            - full_df: DataFrame with all indicator columns appended
    """
    df = df.copy()

    # compute all technical indicators
    # Moving averages
    df['SMA_10'] = ta.sma(df['Close'], length=10)
    df['SMA_20'] = ta.sma(df['Close'], length=20)
    df['SMA_50'] = ta.sma(df['Close'], length=50)
    df['EMA_12'] = ta.ema(df['Close'], length=12)
    df['EMA_26'] = ta.ema(df['Close'], length=26)

    # RSI (Relative Strength Index)
    df['RSI'] = ta.rsi(df['Close'], length=14)

    # MACD (Moving Average Convergence Divergence)
    macd_df = ta.macd(df['Close'], fast=12, slow=26, signal=9)
    df['MACD'] = macd_df['MACD_12_26_9']
    df['MACD_Signal'] = macd_df['MACDs_12_26_9']
    df['MACD_Hist'] = macd_df['MACDh_12_26_9']

    # Bollinger Bands
    bbands = ta.bbands(df['Close'], length=20, std=2)
    for col in bbands.columns:
      if 'BBL' in col:
          df['BB_Lower'] = bbands[col]
      elif 'BBM' in col:
          df['BB_Middle'] = bbands[col]
      elif 'BBU' in col:
          df['BB_Upper'] = bbands[col]
    df['BB_Width'] = (df['BB_Upper'] - df['BB_Lower']) / df['BB_Middle']

    # Stochastic Oscillator
    stoch = ta.stoch(df['High'], df['Low'], df['Close'], k=14, d=3)
    df['Stoch_K'] = stoch['STOCHk_14_3_3']
    df['Stoch_D'] = stoch['STOCHd_14_3_3']
    # ADX (Average Directional Index - trend strength)
    adx_df = ta.adx(df['High'], df['Low'], df['Close'], length=14)
    df['ADX'] = adx_df['ADX_14']
    df['DI_Plus'] = adx_df['DMP_14']
    df['DI_Minus'] = adx_df['DMN_14']

    # ATR (Average True Range)
    df['ATR'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)

    # OBV (On-Balance Volume)
    df['OBV'] = ta.obv(df['Close'], df['Volume'])

    # Momentum & Rate of Change
    df['Momentum'] = ta.mom(df['Close'], length=10)
    df['ROC'] = ta.roc(df['Close'], length=10)

    # extract latest values
    latest = df.iloc[-1]
    prev = df.iloc[-2]

    current_price = float(latest['Close'])

    indicators = {
    "current_price": current_price,
    # Moving averages
    "sma_10": float(latest['SMA_10']) if pd.notna(latest['SMA_10']) else None,
    "sma_20": float(latest['SMA_20']) if pd.notna(latest['SMA_20']) else None,
    "sma_50": float(latest['SMA_50']) if pd.notna(latest['SMA_50']) else None,
    "ema_12": float(latest['EMA_12']) if pd.notna(latest['EMA_12']) else None,
    "ema_26": float(latest['EMA_26']) if pd.notna(latest['EMA_26']) else None,
    # Momentum indicators
    "rsi": float(latest['RSI']) if pd.notna(latest['RSI']) else None,
    "macd": float(latest['MACD']) if pd.notna(latest['MACD']) else None,
    "macd_signal": float(latest['MACD_Signal']) if pd.notna(latest['MACD_Signal']) else None,
    "macd_histogram": float(latest['MACD_Hist']) if pd.notna(latest['MACD_Hist']) else None,
    "stoch_k": float(latest['Stoch_K']) if pd.notna(latest['Stoch_K']) else None,
    "stoch_d": float(latest['Stoch_D']) if pd.notna(latest['Stoch_D']) else None,
    # Volatility indicators
    "bb_upper": float(latest['BB_Upper']) if pd.notna(latest['BB_Upper']) else None,
    "bb_middle": float(latest['BB_Middle']) if pd.notna(latest['BB_Middle']) else None,
    "bb_lower": float(latest['BB_Lower']) if pd.notna(latest['BB_Lower']) else None,
    "bb_width": float(latest['BB_Width']) if pd.notna(latest['BB_Width']) else None,
    "atr": float(latest['ATR']) if pd.notna(latest['ATR']) else None,
    # Trend strength
    "adx": float(latest['ADX']) if pd.notna(latest['ADX']) else None,
    "di_plus": float(latest['DI_Plus']) if pd.notna(latest['DI_Plus']) else None,
    "di_minus": float(latest['DI_Minus']) if pd.notna(latest['DI_Minus']) else None,
    # Other
    "momentum": float(latest['Momentum']) if pd.notna(latest['Momentum']) else None,
    "roc": float(latest['ROC']) if pd.notna(latest['ROC']) else None,
    }

    # rule-based signal interpretation
    signals = {}
    bullish_count = 0
    bearish_count = 0
    # RSI signal
    if indicators["rsi"] is not None:
      if indicators["rsi"] < 30:
        signals["rsi"] = {"signal": "oversold", "direction": "bullish",
                      "detail": f"RSI={indicators['rsi']:.1f}, below 30 -> oversold, potential bounce"}
        bullish_count += 1
    elif indicators["rsi"] > 70:
        signals["rsi"] = {"signal": "overbought", "direction": "bearish",
                      "detail": f"RSI={indicators['rsi']:.1f}, above 70 -> overbought, potential pullback"}
        bearish_count += 1
    else:
        signals["rsi"] = {"signal": "neutral", "direction": "neutral",
                      "detail": f"RSI={indicators['rsi']:.1f}, in neutral zone (30-70)"}

    # MACD signal
    if indicators["macd"] is not None and indicators["macd_signal"] is not None:
        macd_cross = indicators["macd"] - indicators["macd_signal"]
        prev_macd_cross = float(prev['MACD'] - prev['MACD_Signal']) if pd.notna(prev['MACD']) else None

        if indicators["macd"] > indicators["macd_signal"]:
            direction = "bullish"
            bullish_count += 1
          # Detect fresh golden cross
            if prev_macd_cross is not None and prev_macd_cross < 0:
                detail = f"MACD just crossed ABOVE signal line -> fresh bullish crossover"
            else:
                detail = f"MACD({indicators['macd']:.2f}) > Signal({indicators['macd_signal']:.2f}) -> bullish"
        else:
            direction = "bearish"
            bearish_count += 1
            if prev_macd_cross is not None and prev_macd_cross > 0:
                detail = f"MACD just crossed BELOW signal line -> fresh bearish crossover"
            else:
                detail = f"MACD({indicators['macd']:.2f}) < Signal({indicators['macd_signal']:.2f}) -> bearish"
        signals["macd"] = {"signal": "bullish_cross" if direction == "bullish" else "bearish_cross",
                  "direction": direction, "detail": detail}

  # Moving average alignment signal
    if all(indicators[k] is not None for k in ["sma_20", "sma_50"]):
        if current_price > indicators["sma_20"] > indicators["sma_50"]:
            signals["moving_averages"] = {"signal": "bullish_alignment", "direction": "bullish",
                                          "detail": f"Price(${current_price:.2f}) > SMA20(${indicators['sma_20']:.2f}) > SMA50(${indicators['sma_50']:.2f})"}
            bullish_count += 1
        elif current_price < indicators["sma_20"] < indicators["sma_50"]:
            signals["moving_averages"] = {"signal": "bearish_alignment", "direction": "bearish",
                                          "detail": f"Price(${current_price:.2f}) < SMA20(${indicators['sma_20']:.2f}) < SMA50(${indicators['sma_50']:.2f})"}
            bearish_count += 1
        else:
            signals["moving_averages"] = {"signal": "mixed", "direction": "neutral",
                                          "detail": f"Mixed: Price=${current_price:.2f}, SMA20=${indicators['sma_20']:.2f}, SMA50=${indicators['sma_50']:.2f}"}

    # Bollinger Bands signal
    if indicators["bb_upper"] is not None and indicators["bb_lower"] is not None:
        if current_price > indicators["bb_upper"]:
            signals["bollinger"] = {"signal": "above_upper", "direction": "bearish",
                                    "detail": f"Price above upper band (${indicators['bb_upper']:.2f}) -> potential reversal down"}
            bearish_count += 1
        elif current_price < indicators["bb_lower"]:
            signals["bollinger"] = {"signal": "below_lower", "direction": "bullish",
                                    "detail": f"Price below lower band (${indicators['bb_lower']:.2f}) -> potential reversal up"}
            bullish_count += 1
        else:
            bb_position = (current_price - indicators["bb_lower"]) / (indicators["bb_upper"] - indicators["bb_lower"])
            signals["bollinger"] = {"signal": "within_bands", "direction": "neutral",
                                    "detail": f"Price within bands, position: {bb_position:.0%} (0%=lower, 100%=upper)"}

    # Stochastic signal
    if indicators["stoch_k"] is not None:
        if indicators["stoch_k"] > 80:
            signals["stochastic"] = {"signal": "overbought", "direction": "bearish",
                                      "detail": f"%K={indicators['stoch_k']:.1f}, above 80 -> overbought"}
            bearish_count += 1
        elif indicators["stoch_k"] < 20:
            signals["stochastic"] = {"signal": "oversold", "direction": "bullish",
                                      "detail": f"%K={indicators['stoch_k']:.1f}, below 20 -> oversold"}
            bullish_count += 1
        else:
            signals["stochastic"] = {"signal": "neutral", "direction": "neutral",
                                      "detail": f"%K={indicators['stoch_k']:.1f}, in neutral zone"}
    # ADX trend strength
    if indicators["adx"] is not None:
        trend_strength = "strong" if indicators["adx"] > 25 else "weak"
        trend_dir = "up" if (indicators.get("di_plus") or 0) > (indicators.get("di_minus") or 0) else "down"
        signals["adx"] = {
            "signal": f"{trend_strength}_trend_{trend_dir}",
            "direction": "bullish" if trend_dir == "up" and trend_strength == "strong" else
                         "bearish" if trend_dir == "down" and trend_strength == "strong" else "neutral",
            "detail": f"ADX={indicators['adx']:.1f} ({'strong trend' if trend_strength == 'strong' else 'weak/no trend'}), "
                      f"+DI={indicators.get('di_plus', 0):.1f}, -DI={indicators.get('di_minus', 0):.1f} -> trend direction: {trend_dir}"
        }
        if trend_strength == "strong":
            if trend_dir == "up":
                bullish_count += 1
            else:
                bearish_count += 1

    # Volume-price divergence detection
    # Price new high + volume decline = bearish; price new low + volume decline = selling exhaustion
    vp_div = _detect_volume_price_divergence(df)
    if vp_div["detected"]:
        signals["volume_divergence"] = {
            "signal": vp_div["type"],
            "direction": vp_div["direction"],
            "detail": vp_div["detail"]
        }
        if vp_div["direction"] == "bullish":
            bullish_count += 1
        elif vp_div["direction"] == "bearish":
            bearish_count += 1

    # OBV trend (On-Balance Volume)
    if 'OBV' in df.columns and len(df) >= 20:
        obv_sma = df['OBV'].rolling(20).mean()
        if pd.notna(obv_sma.iloc[-1]):
            obv_current = float(df['OBV'].iloc[-1])
            obv_avg = float(obv_sma.iloc[-1])
            if obv_current > obv_avg * 1.05:
                signals["obv"] = {"signal": "obv_rising", "direction": "bullish",
                                  "detail": f"OBV above 20-day average -> buying pressure increasing"}
                bullish_count += 1
            elif obv_current < obv_avg * 0.95:
                signals["obv"] = {"signal": "obv_falling", "direction": "bearish",
                                  "detail": f"OBV below 20-day average -> selling pressure increasing"}
                bearish_count += 1
            else:
                signals["obv"] = {"signal": "obv_flat", "direction": "neutral",
                                  "detail": f"OBV near 20-day average -> no clear volume trend"}

    signal_stats = {}
    if 'RSI' in df.columns:
        signal_stats["rsi_overbought_days"] = int((df['RSI'] > 70).sum())
        signal_stats["rsi_oversold_days"] = int((df['RSI'] < 30).sum())

    if 'MACD' in df.columns and 'MACD_Signal' in df.columns:
        macd_diff = df['MACD'] - df['MACD_Signal']
        signal_stats["macd_golden_crosses"] = int(((macd_diff > 0) & (macd_diff.shift(1) < 0)).sum())
        signal_stats["macd_death_crosses"] = int(((macd_diff < 0) & (macd_diff.shift(1) > 0)).sum())

    if 'BB_Upper' in df.columns and 'BB_Lower' in df.columns:
        signal_stats["bb_above_upper_days"] = int((df['Close'] > df['BB_Upper']).sum())
        signal_stats["bb_below_lower_days"] = int((df['Close'] < df['BB_Lower']).sum())

    # summary
    total = bullish_count + bearish_count
    if total > 0:
        bullish_pct = bullish_count / total * 100
    else:
        bullish_pct = 50

    signal_summary = (
        f"Technical signals: {bullish_count} bullish, {bearish_count} bearish, "
        f"{len(signals) - bullish_count - bearish_count} neutral. "
        f"Overall lean: {'BULLISH' if bullish_count > bearish_count else 'BEARISH' if bearish_count > bullish_count else 'NEUTRAL'} "
        f"({bullish_pct:.0f}% bullish)"
    )

    return {
        "indicators": indicators,
        "signals": signals,
        "signal_stats": signal_stats,
        "signal_summary": signal_summary,
        "bullish_count": bullish_count,
        "bearish_count": bearish_count,
        "full_df": df,
    }



In [ ]:
# option chain agent
def get_options_data(ticker: str, num_expirations: int = 3) -> dict:
    """
    Fetch options chain data and compute derived metrics.

    Parameters:
        ticker: Stock symbol
        num_expirations: Number of nearest expiration dates to fetch, default 3

    Returns:
        dict with keys:
            - ticker: str
            - current_price: float
            - expirations: list of expiration dates used
            - put_call_ratio: float
            - max_pain: float (nearest expiration)
            - max_call_oi_strike: float (resistance level)
            - max_put_oi_strike: float (support level)
            - atm_iv: float (at-the-money implied volatility)
            - iv_skew: float (OTM put IV - ATM IV)
            - iv_term_structure: dict (near-term vs far-term IV)
            - call_total_oi: int
            - put_total_oi: int
            - chains: dict of raw chain DataFrames
    """
    try:
        stock = yf.Ticker(ticker)
        current_price = stock.info.get('currentPrice') or stock.info.get('regularMarketPrice')

        # Fallback: get price from historical data
        if current_price is None:
            hist = stock.history(period="1d")
            if not hist.empty:
                current_price = float(hist['Close'].iloc[-1])
            else:
                return {"error": f"Cannot get current price for {ticker}"}

        current_price = float(current_price)
    except Exception as e:
        return {"error": f"Failed to get ticker info: {str(e)}"}

    # Fetch options chains
    try:
        available_exps = stock.options
    except Exception as e:
        return {"error": f"No options data available for {ticker}: {str(e)}"}

    if not available_exps:
        return {"error": f"No options expirations found for {ticker}"}

    expirations = list(available_exps[:num_expirations])

    all_calls = []
    all_puts = []
    chains = {}

    for exp in expirations:
        try:
            chain = stock.option_chain(exp)
            calls = chain.calls.copy()
            puts = chain.puts.copy()

            # Add expiration column
            calls['expiration'] = exp
            puts['expiration'] = exp

            # Clean: remove rows where openInterest is NaN or 0
            calls = calls[calls['openInterest'].notna() & (calls['openInterest'] > 0)]
            puts = puts[puts['openInterest'].notna() & (puts['openInterest'] > 0)]

            all_calls.append(calls)
            all_puts.append(puts)
            chains[exp] = {"calls": chain.calls, "puts": chain.puts}  # Keep raw data
        except Exception as e:
            print(f"Warning: failed to get chain for {exp}: {e}")
            continue

    if not all_calls or not all_puts:
        return {"error": "Failed to retrieve any valid options data"}

    calls_df = pd.concat(all_calls, ignore_index=True)
    puts_df = pd.concat(all_puts, ignore_index=True)

    # Put/Call Ratio
    call_total_oi = int(calls_df['openInterest'].sum())
    put_total_oi = int(puts_df['openInterest'].sum())
    put_call_ratio = round(put_total_oi / call_total_oi, 4) if call_total_oi > 0 else None

    # Max Pain (nearest expiration)
    nearest_exp = expirations[0]
    nearest_calls = calls_df[calls_df['expiration'] == nearest_exp]
    nearest_puts = puts_df[puts_df['expiration'] == nearest_exp]
    max_pain = _calculate_max_pain(nearest_calls, nearest_puts, current_price)

    # Max OI Strike (support/resistance levels)
    max_call_oi_strike = None
    max_put_oi_strike = None

    if not nearest_calls.empty:
        max_call_oi_idx = nearest_calls['openInterest'].idxmax()
        max_call_oi_strike = float(nearest_calls.loc[max_call_oi_idx, 'strike'])

    if not nearest_puts.empty:
        max_put_oi_idx = nearest_puts['openInterest'].idxmax()
        max_put_oi_strike = float(nearest_puts.loc[max_put_oi_idx, 'strike'])

    # ATM Implied Volatility
    atm_iv = _get_atm_iv(nearest_calls, nearest_puts, current_price)

    # IV Skew
    iv_skew = _calculate_iv_skew(nearest_puts, current_price, atm_iv)

    # IV Term Structure (near-term vs far-term)
    iv_term_structure = _calculate_iv_term_structure(calls_df, puts_df, expirations, current_price)

    # Signal interpretation
    options_signals = {}

    if put_call_ratio is not None:
        if put_call_ratio > 1.2:
            options_signals["put_call_ratio"] = {
                "signal": "high_put_call", "direction": "bearish",
                "detail": f"P/C Ratio={put_call_ratio:.2f} (>1.2) -> heavy put buying, market is fearful"
            }
        elif put_call_ratio < 0.7:
            options_signals["put_call_ratio"] = {
                "signal": "low_put_call", "direction": "bullish",
                "detail": f"P/C Ratio={put_call_ratio:.2f} (<0.7) -> heavy call buying, market is optimistic"
            }
        else:
            options_signals["put_call_ratio"] = {
                "signal": "neutral_put_call", "direction": "neutral",
                "detail": f"P/C Ratio={put_call_ratio:.2f} -> balanced sentiment"
            }

    if max_pain is not None:
        pain_diff_pct = (max_pain - current_price) / current_price * 100
        if pain_diff_pct > 3:
            options_signals["max_pain"] = {
                "signal": "max_pain_above", "direction": "bullish",
                "detail": f"Max Pain=${max_pain:.2f} is {pain_diff_pct:.1f}% above current price -> potential upward pull"
            }
        elif pain_diff_pct < -3:
            options_signals["max_pain"] = {
                "signal": "max_pain_below", "direction": "bearish",
                "detail": f"Max Pain=${max_pain:.2f} is {abs(pain_diff_pct):.1f}% below current price -> potential downward pull"
            }
        else:
            options_signals["max_pain"] = {
                "signal": "max_pain_near", "direction": "neutral",
                "detail": f"Max Pain=${max_pain:.2f} is near current price (${current_price:.2f}) -> price likely pinned"
            }

    if iv_skew is not None:
        if iv_skew > 0.1:
            options_signals["iv_skew"] = {
                "signal": "high_skew", "direction": "bearish",
                "detail": f"IV Skew={iv_skew:.3f} -> elevated demand for downside protection"
            }
        else:
            options_signals["iv_skew"] = {
                "signal": "normal_skew", "direction": "neutral",
                "detail": f"IV Skew={iv_skew:.3f} -> normal skew levels"
            }

    return {
        "ticker": ticker,
        "current_price": current_price,
        "expirations": expirations,
        # Core metrics
        "put_call_ratio": put_call_ratio,
        "max_pain": max_pain,
        "max_call_oi_strike": max_call_oi_strike,  # Resistance level
        "max_put_oi_strike": max_put_oi_strike,      # Support level
        "atm_iv": atm_iv,
        "iv_skew": iv_skew,
        "iv_term_structure": iv_term_structure,
        # Total open interest
        "call_total_oi": call_total_oi,
        "put_total_oi": put_total_oi,
        # Signals
        "options_signals": options_signals,
        # Raw data (for debug or further analysis)
        "chains": chains,
    }

In [ ]:
# options helper functions
def _calculate_max_pain(calls: pd.DataFrame, puts: pd.DataFrame, current_price: float) -> float:
    """
    Calculate Max Pain: the strike price that minimizes total option holders' intrinsic value.
    For each possible expiration price (strike), compute total intrinsic value of all contracts,
    and find the strike that minimizes this total.
    """
    if calls.empty and puts.empty:
        return None

    # Collect all strikes
    all_strikes = sorted(set(
        list(calls['strike'].unique()) + list(puts['strike'].unique())
    ))

    if not all_strikes:
        return None

    min_pain = float('inf')
    max_pain_strike = all_strikes[0]

    for test_price in all_strikes:
        total_pain = 0

        # Call intrinsic value: max(test_price - strike, 0) * openInterest
        if not calls.empty:
            call_pain = calls.apply(
                lambda row: max(test_price - row['strike'], 0) * row['openInterest'],
                axis=1
            ).sum()
            total_pain += call_pain

        # Put intrinsic value: max(strike - test_price, 0) * openInterest
        if not puts.empty:
            put_pain = puts.apply(
                lambda row: max(row['strike'] - test_price, 0) * row['openInterest'],
                axis=1
            ).sum()
            total_pain += put_pain

        if total_pain < min_pain:
            min_pain = total_pain
            max_pain_strike = test_price

    return float(max_pain_strike)


def _get_atm_iv(calls: pd.DataFrame, puts: pd.DataFrame, current_price: float) -> float:
    """Get implied volatility of the strike nearest to current price (average of call and put)."""
    atm_iv = None

    try:
        if not calls.empty:
            calls_valid = calls[calls['impliedVolatility'].notna() & (calls['impliedVolatility'] > 0)]
            if not calls_valid.empty:
                atm_call_idx = (calls_valid['strike'] - current_price).abs().idxmin()
                atm_call_iv = float(calls_valid.loc[atm_call_idx, 'impliedVolatility'])
            else:
                atm_call_iv = None
        else:
            atm_call_iv = None

        if not puts.empty:
            puts_valid = puts[puts['impliedVolatility'].notna() & (puts['impliedVolatility'] > 0)]
            if not puts_valid.empty:
                atm_put_idx = (puts_valid['strike'] - current_price).abs().idxmin()
                atm_put_iv = float(puts_valid.loc[atm_put_idx, 'impliedVolatility'])
            else:
                atm_put_iv = None
        else:
            atm_put_iv = None

        # Average of available IVs
        ivs = [v for v in [atm_call_iv, atm_put_iv] if v is not None]
        if ivs:
            atm_iv = round(sum(ivs) / len(ivs), 6)

    except Exception:
        pass

    return atm_iv


def _calculate_iv_skew(puts: pd.DataFrame, current_price: float, atm_iv: float) -> float:
    """
    Calculate IV Skew: difference between average OTM put IV and ATM IV.
    OTM puts = puts with strike below current price.
    Skew > 0 indicates higher demand for downside protection.
    """
    if atm_iv is None or puts.empty:
        return None

    try:
        otm_puts = puts[
            (puts['strike'] < current_price * 0.95) &  # At least 5% OTM
            (puts['impliedVolatility'].notna()) &
            (puts['impliedVolatility'] > 0)
        ]

        if otm_puts.empty:
            return None

        avg_otm_iv = float(otm_puts['impliedVolatility'].mean())
        return round(avg_otm_iv - atm_iv, 6)

    except Exception:
        return None


def _calculate_iv_term_structure(
    calls_df: pd.DataFrame, puts_df: pd.DataFrame,
    expirations: list, current_price: float
) -> dict:
    """
    Calculate IV term structure: compare ATM IV across expirations.
    If near-term IV > far-term IV (backwardation), short-term uncertainty is elevated.
    """
    if len(expirations) < 2:
        return {"available": False}

    term_ivs = {}

    for exp in expirations:
        exp_calls = calls_df[calls_df['expiration'] == exp]
        exp_puts = puts_df[puts_df['expiration'] == exp]
        iv = _get_atm_iv(exp_calls, exp_puts, current_price)
        if iv is not None:
            term_ivs[exp] = iv

    if len(term_ivs) < 2:
        return {"available": False}

    sorted_exps = sorted(term_ivs.keys())
    near_iv = term_ivs[sorted_exps[0]]
    far_iv = term_ivs[sorted_exps[-1]]

    return {
        "available": True,
        "by_expiration": term_ivs,
        "near_term_iv": near_iv,
        "far_term_iv": far_iv,
        "structure": "backwardation" if near_iv > far_iv else "contango",
        "detail": f"Near({sorted_exps[0]}) IV={near_iv:.3f}, Far({sorted_exps[-1]}) IV={far_iv:.3f} -> "
                  f"{'short-term uncertainty is higher' if near_iv > far_iv else 'normal term structure'}"
    }

In [ ]:
# weighted composite score - time decay weighted scoring
def get_weighted_signals(tech_result: dict, options_result: dict = None,
                         recent_days: int = 5) -> dict:
    """
    Apply time-decay weighting to all signals and compute a composite score.

    Inspired by the layered memory mechanism in the FinMem paper:
    - Signals from the most recent N days get 2x weight (short-term memory)
    - Older signals get 1x weight (long-term memory)
    - Different signal categories have different base weights (technical vs options)

    Parameters:
        tech_result: Return value from get_technical_indicators()
        options_result: Return value from get_options_data() (optional)
        recent_days: Number of days in the recency window, default 5

    Returns:
        dict with keys:
            - composite_score: float (-100 to 100, positive=bullish, negative=bearish)
            - confidence: float (0-1, higher when signals agree)
            - recent_momentum: str ("accelerating_bullish"/"decelerating"/...)
            - signal_agreement: float (signal consistency percentage)
            - score_breakdown: dict (sub-scores by category)
            - recommendation_strength: str ("strong"/"moderate"/"weak")
    """
    df = tech_result.get("full_df")
    if df is None or df.empty:
        return {"error": "No data available for weighted scoring"}

    # 1. Time-decay weighted technical signals
    recent = df.tail(recent_days)
    older = df.iloc[:-recent_days] if len(df) > recent_days else pd.DataFrame()

    tech_score = 0.0

    # RSI with time decay
    if 'RSI' in df.columns:
        recent_rsi = recent['RSI'].dropna()
        older_rsi = older['RSI'].dropna() if not older.empty else pd.Series(dtype=float)

        # Recent oversold days (weight 2x) + older oversold days (weight 1x)
        recent_oversold = (recent_rsi < 30).sum() * 2
        recent_overbought = (recent_rsi > 70).sum() * 2
        older_oversold = (older_rsi < 30).sum() * 1 if not older_rsi.empty else 0
        older_overbought = (older_rsi > 70).sum() * 1 if not older_rsi.empty else 0

        tech_score += (recent_oversold + older_oversold) * 5
        tech_score -= (recent_overbought + older_overbought) * 5

    # MACD with time decay: recent crossovers have higher weight
    if 'MACD' in df.columns and 'MACD_Signal' in df.columns:
        recent_macd_diff = (recent['MACD'] - recent['MACD_Signal']).dropna()
        if not recent_macd_diff.empty:
            recent_golden = ((recent_macd_diff > 0) & (recent_macd_diff.shift(1) < 0)).sum()
            recent_death = ((recent_macd_diff < 0) & (recent_macd_diff.shift(1) > 0)).sum()
            tech_score += recent_golden * 15  # Recent golden cross gets high weight
            tech_score -= recent_death * 15

        # Current MACD direction
        if not recent_macd_diff.empty and recent_macd_diff.iloc[-1] > 0:
            tech_score += 5
        elif not recent_macd_diff.empty:
            tech_score -= 5

    # Moving average position
    indicators = tech_result.get("indicators", {})
    price = indicators.get("current_price", 0)
    sma20 = indicators.get("sma_20")
    sma50 = indicators.get("sma_50")
    if price and sma20 and sma50:
        if price > sma20 > sma50:
            tech_score += 10
        elif price < sma20 < sma50:
            tech_score -= 10

    # Bollinger Bands position
    bb_lower = indicators.get("bb_lower")
    bb_upper = indicators.get("bb_upper")
    if price and bb_lower and bb_upper:
        if price < bb_lower:
            tech_score += 8  # Oversold bounce
        elif price > bb_upper:
            tech_score -= 8  # Overbought pullback

    # ADX trend strength bonus
    adx = indicators.get("adx")
    di_plus = indicators.get("di_plus", 0)
    di_minus = indicators.get("di_minus", 0)
    if adx and adx > 25:
        trend_bonus = 5
        if di_plus > di_minus:
            tech_score += trend_bonus
        else:
            tech_score -= trend_bonus

    # 2. Options signal weighting
    options_score = 0.0
    if options_result and "error" not in options_result:
        pcr = options_result.get("put_call_ratio")
        if pcr is not None:
            if pcr > 1.2:
                options_score -= 10  # Market fear -> bearish
            elif pcr < 0.7:
                options_score += 10  # Market optimism -> bullish
            # Contrarian logic: extreme fear may signal a bottom
            if pcr > 1.5:
                options_score += 5  # Extreme fear -> contrarian bullish

        max_pain = options_result.get("max_pain")
        curr_price = options_result.get("current_price")
        if max_pain and curr_price:
            pain_pct = (max_pain - curr_price) / curr_price * 100
            if pain_pct > 3:
                options_score += 8
            elif pain_pct < -3:
                options_score -= 8

        iv_skew = options_result.get("iv_skew")
        if iv_skew is not None and iv_skew > 0.1:
            options_score -= 5  # High skew -> fear of downside

    # 3. Recent momentum assessment
    recent_momentum = "neutral"
    if 'RSI' in df.columns and len(recent) >= 2:
        rsi_now = recent['RSI'].iloc[-1] if pd.notna(recent['RSI'].iloc[-1]) else None
        rsi_prev = recent['RSI'].iloc[0] if pd.notna(recent['RSI'].iloc[0]) else None
        if rsi_now is not None and rsi_prev is not None:
            rsi_change = rsi_now - rsi_prev
            if rsi_change > 5 and rsi_now > 50:
                recent_momentum = "accelerating_bullish"
            elif rsi_change > 5 and rsi_now <= 50:
                recent_momentum = "recovering"
            elif rsi_change < -5 and rsi_now < 50:
                recent_momentum = "accelerating_bearish"
            elif rsi_change < -5 and rsi_now >= 50:
                recent_momentum = "weakening"
            else:
                recent_momentum = "steady"

    # 4. Composite score
    # Technical weight 60%, options weight 40%
    composite = tech_score * 0.6 + options_score * 0.4
    # Clamp to [-100, 100]
    composite = max(-100, min(100, composite))

    # 5. Signal agreement (confidence)
    all_signals = tech_result.get("signals", {})
    directions = [s["direction"] for s in all_signals.values() if s["direction"] != "neutral"]
    if options_result and "error" not in options_result:
        opt_sigs = options_result.get("options_signals", {})
        directions += [s["direction"] for s in opt_sigs.values() if s["direction"] != "neutral"]

    if directions:
        bullish_ratio = directions.count("bullish") / len(directions)
        bearish_ratio = directions.count("bearish") / len(directions)
        agreement = max(bullish_ratio, bearish_ratio)
    else:
        agreement = 0.5

    confidence = round(agreement, 3)

    abs_score = abs(composite)
    if abs_score > 40 and confidence > 0.6:
        strength = "strong"
    elif abs_score > 20:
        strength = "moderate"
    else:
        strength = "weak"

    return {
        "composite_score": round(composite, 2),
        "confidence": confidence,
        "recent_momentum": recent_momentum,
        "signal_agreement": round(agreement * 100, 1),
        "score_breakdown": {
            "technical_score": round(tech_score, 2),
            "options_score": round(options_score, 2),
            "tech_weight": 0.6,
            "options_weight": 0.4,
        },
        "recommendation_strength": strength,
        "direction": "bullish" if composite > 10 else "bearish" if composite < -10 else "neutral",
    }


In [ ]:
def get_support_resistance(df: pd.DataFrame, options_result: dict = None,
                           window: int = 20) -> dict:
    """
    Compute support and resistance levels from multiple data sources.

    Sources:
    1. Technical: recent highs/lows, moving averages, Bollinger Bands
    2. Options: max pain, highest OI strikes
    3. Volume clusters (simplified Volume Profile)

    Parameters:
        df: DataFrame with OHLCV and technical indicator columns
        options_result: Return value from get_options_data() (optional)
        window: Lookback window in trading days

    Returns:
        dict with keys:
            - support_levels: list of (price, source, strength) tuples
            - resistance_levels: list of (price, source, strength) tuples
            - key_support: float (strongest support level)
            - key_resistance: float (strongest resistance level)
    """
    current_price = float(df['Close'].iloc[-1])
    supports = []
    resistances = []

    recent = df.tail(window)
    recent_high = float(recent['High'].max())
    recent_low = float(recent['Low'].min())

    if recent_low < current_price:
        supports.append({"price": recent_low, "source": "recent_low", "strength": 2})
    if recent_high > current_price:
        resistances.append({"price": recent_high, "source": "recent_high", "strength": 2})

    for col, name in [('SMA_20', 'SMA20'), ('SMA_50', 'SMA50'), ('EMA_12', 'EMA12')]:
        if col in df.columns and pd.notna(df[col].iloc[-1]):
            ma_val = float(df[col].iloc[-1])
            if ma_val < current_price:
                supports.append({"price": ma_val, "source": name, "strength": 2 if 'SMA_50' == col else 1})
            elif ma_val > current_price:
                resistances.append({"price": ma_val, "source": name, "strength": 2 if 'SMA_50' == col else 1})


    if 'BB_Lower' in df.columns and pd.notna(df['BB_Lower'].iloc[-1]):
        bb_lower = float(df['BB_Lower'].iloc[-1])
        bb_upper = float(df['BB_Upper'].iloc[-1])
        supports.append({"price": bb_lower, "source": "BB_lower", "strength": 1})
        resistances.append({"price": bb_upper, "source": "BB_upper", "strength": 1})

    if len(recent) >= 10:
        price_bins = pd.cut(recent['Close'], bins=10)
        vol_by_price = recent.groupby(price_bins, observed=True)['Volume'].sum()
        if not vol_by_price.empty:
            top_vol_bin = vol_by_price.idxmax()
            vol_center = (top_vol_bin.left + top_vol_bin.right) / 2
            if vol_center < current_price:
                supports.append({"price": round(float(vol_center), 2),
                                "source": "volume_cluster", "strength": 3})
            elif vol_center > current_price:
                resistances.append({"price": round(float(vol_center), 2),
                                   "source": "volume_cluster", "strength": 3})


    if options_result and "error" not in options_result:
        max_pain = options_result.get("max_pain")
        if max_pain:
            if max_pain < current_price:
                supports.append({"price": max_pain, "source": "max_pain", "strength": 3})
            elif max_pain > current_price:
                resistances.append({"price": max_pain, "source": "max_pain", "strength": 3})

        put_strike = options_result.get("max_put_oi_strike")
        if put_strike and put_strike < current_price:
            supports.append({"price": put_strike, "source": "put_wall", "strength": 3})

        call_strike = options_result.get("max_call_oi_strike")
        if call_strike and call_strike > current_price:
            resistances.append({"price": call_strike, "source": "call_wall", "strength": 3})

    supports = _merge_nearby_levels(supports, threshold_pct=0.01)
    resistances = _merge_nearby_levels(resistances, threshold_pct=0.01)

    supports.sort(key=lambda x: x["strength"], reverse=True)
    resistances.sort(key=lambda x: x["strength"], reverse=True)

    return {
        "current_price": current_price,
        "support_levels": supports,
        "resistance_levels": resistances,
        "key_support": supports[0]["price"] if supports else None,
        "key_resistance": resistances[0]["price"] if resistances else None,
    }


def _merge_nearby_levels(levels: list, threshold_pct: float = 0.01) -> list:
    """Merge support/resistance levels that are within +/-threshold_pct of each other."""
    if not levels:
        return []

    sorted_levels = sorted(levels, key=lambda x: x["price"])
    merged = [sorted_levels[0].copy()]

    for level in sorted_levels[1:]:
        if abs(level["price"] - merged[-1]["price"]) / merged[-1]["price"] < threshold_pct:
            total_strength = merged[-1]["strength"] + level["strength"]
            merged[-1]["price"] = (
                merged[-1]["price"] * merged[-1]["strength"] +
                level["price"] * level["strength"]
            ) / total_strength
            merged[-1]["price"] = round(merged[-1]["price"], 2)
            merged[-1]["strength"] = total_strength
            merged[-1]["source"] = f"{merged[-1]['source']}+{level['source']}"
        else:
            merged.append(level.copy())

    return merged


In [ ]:
def _calculate_sortino(returns: pd.Series) -> float:
    """Calculate Sortino Ratio: only penalizes downside volatility (more suitable than Sharpe for risk)."""
    downside = returns[returns < 0]
    if downside.empty or downside.std() == 0:
        return 0.0
    return float((returns.mean() / downside.std()) * np.sqrt(252))


def _detect_volume_price_divergence(df: pd.DataFrame, lookback: int = 10) -> dict:
    """
    Detect volume-price divergence:
    - Bearish divergence: price makes new high but volume declines (upward momentum fading)
    - Bullish divergence: price makes new low but volume declines (selling exhaustion)
    """
    if len(df) < lookback + 5:
        return {"detected": False}

    recent = df.tail(lookback)
    prev = df.iloc[-(lookback * 2):-lookback]

    if prev.empty:
        return {"detected": False}

    recent_high = recent['Close'].max()
    prev_high = prev['Close'].max()
    recent_avg_vol = recent['Volume'].mean()
    prev_avg_vol = prev['Volume'].mean()

    recent_low = recent['Close'].min()
    prev_low = prev['Close'].min()

    if recent_high > prev_high and recent_avg_vol < prev_avg_vol * 0.85:
        return {
            "detected": True,
            "type": "bearish_divergence",
            "direction": "bearish",
            "detail": f"Price made new high (${recent_high:.2f} > ${prev_high:.2f}) "
                      f"but volume declined ({recent_avg_vol/1e6:.1f}M vs {prev_avg_vol/1e6:.1f}M) -> bearish divergence"
        }

    if recent_low < prev_low and recent_avg_vol < prev_avg_vol * 0.85:
        return {
            "detected": True,
            "type": "bullish_divergence",
            "direction": "bullish",
            "detail": f"Price made new low (${recent_low:.2f} < ${prev_low:.2f}) "
                      f"but volume declined ({recent_avg_vol/1e6:.1f}M vs {prev_avg_vol/1e6:.1f}M) -> selling exhaustion"
        }

    return {"detected": False}

**Rebecca:**

In [ ]:
from typing import Dict, Any
from openai import OpenAI

def planner(state: State) -> State:
    """
    Input: state
    Output: updated state with ticker, task, and user_profile
    """

    query = state.get("query", "")
    user_id = state.get("user_id", "default_user")

    state["user_profile"] = get_user_profile(user_id)

    try:
        prompt = f"""
You are a trading assistant.

Extract structured information from the query.

Query:
"{query}"

Return JSON:
{{
  "ticker": "stock ticker if mentioned, else null",
  "task": one of ["buy", "sell", "roll_option", "general"]
}}
"""
        response = client.chat.completions.create(
            model="gpt-5.3",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0
        )

        result = json.loads(response.choices[0].message.content)

    except Exception:
        result = {"ticker": None, "task": "general"}

    ticker = result.get("ticker")
    task = result.get("task", "general")

    if ticker:
        state["ticker"] = ticker.upper()

    state["task"] = task

    state.setdefault("market_data", {})
    state.setdefault("technical_signals", {})
    state.setdefault("options_data", {})
    state.setdefault("price_data", None)
    state.setdefault("technical_full", {})
    state.setdefault("options_full", {})
    state.setdefault("news_sentiment", {})
    state.setdefault("risk_score", 0.0)
    state.setdefault("weighted_signals", {})
    state.setdefault("llm_analysis", "")
    state.setdefault("decision", {})

    return state

NameError: name 'State' is not defined

In [ ]:
def decision_agent(state: State) -> State:
    quant = state.get("weighted_signals", {})
    risk = state.get("risk_score", 0.5)
    news = state.get("news_sentiment", {})
    tech = state.get("technical_signals", {})
    user = state.get("user_profile", {})

    prompt = f"""
You are a professional trading assistant.

User risk profile: {user.get("risk_profile")}

Quant signals:
- composite_score: {quant.get("composite_score")}
- confidence: {quant.get("confidence")}

Technical:
- RSI: {tech.get("rsi")}

News sentiment:
- score: {news.get("score")}
- summary: {news.get("summary")}

Risk score: {risk}

Question: {state.get("query")}

Tasks:
1. Give final action: BUY / SELL / HOLD / ROLL
2. Give reasoning
3. Give confidence (0-1)
4. Adjust recommendation based on user risk profile

Return JSON:
{{
  "action": "...",
  "reason": "...",
  "confidence": ...
}}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    import json
    result = json.loads(response.choices[0].message.content)

    state["decision"] = result
    state["llm_analysis"] = response.choices[0].message.content

    return state

In [ ]:
import random
import feedparser

def news_agent(state: State) -> State:
    """
    Input: state
    Output: updated state with news_sentiment (real headlines via Google RSS + sentiment)
    """

    ticker = state.get("ticker", "UNKNOWN")
    if ticker == "UNKNOWN":
        state["news_sentiment"] = {
            "score": 0.0,
            "summary": "No ticker provided, cannot fetch news."
        }
        return state

    rss_url = f"https://news.google.com/rss/search?q={ticker}+stock&hl=en-US&gl=US&ceid=US:en"

    feed = feedparser.parse(rss_url)
    headlines = [entry.title for entry in feed.entries[:5]]
    if headlines:
        sentiment_score = random.uniform(-0.5, 0.5)

        if sentiment_score > 0.2:
            summary = f"Recent news for {ticker} is generally positive."
        elif sentiment_score < -0.2:
            summary = f"Recent news for {ticker} is generally negative."
        else:
            summary = f"Recent news for {ticker} is mixed."

    else:
        sentiment_score = 0.0
        summary = f"No recent news found for {ticker}."

    state["news_sentiment"] = {
        "score": round(sentiment_score, 3),
        "summary": summary,
        "headlines": headlines
    }

    return state

ModuleNotFoundError: No module named 'feedparser'

**Alycia:**

In [ ]:
def market_agent(state: State) -> State:
    result = get_market_data(state["ticker"])

    if "error" in result:
        state["market_data"] = {}
        return state

    stats = result["stats"]

    state["market_data"] = {
        "volatility": stats.get("annualized_volatility"),
        "returns": stats.get("avg_daily_return"),
        "drawdown": stats.get("max_drawdown"),
        "sharpe": stats.get("sharpe_ratio"),
    }

    state["price_data"] = result.get("price_data")

    return state

In [ ]:
def technical_agent(state: State) -> State:
    df = state.get("price_data")

    if df is None:
        state["technical_signals"] = {}
        return state

    result = get_technical_indicators(df)

    indicators = result.get("indicators", {})

    state["technical_signals"] = {
        "rsi": indicators.get("rsi"),
        "macd": indicators.get("macd"),
        "adx": indicators.get("adx"),
        "bullish_count": result.get("bullish_count", 0),
        "bearish_count": result.get("bearish_count", 0),
        "summary": result.get("signal_summary"),
    }

    state["technical_full"] = result

    return state

In [ ]:
def options_agent(state: State) -> State:
    result = get_options_data(state["ticker"])

    if "error" in result:
        state["options_data"] = {}
        return state

    state["options_data"] = {
        "implied_volatility": result.get("atm_iv"),
        "put_call_ratio": result.get("put_call_ratio"),
        "iv_skew": result.get("iv_skew"),
        "max_pain": result.get("max_pain"),
    }

    state["options_full"] = result

    return state

In [ ]:
def risk_agent(state: State) -> State:
    tech = state.get("technical_full")
    options = state.get("options_full")

    if not tech:
        state["risk_score"] = 0.5
        return state

    weighted = get_weighted_signals(tech, options)

    composite = weighted.get("composite_score", 0)

    risk_score = 1 - (composite + 100) / 200

    state["risk_score"] = round(float(risk_score), 3)
    state["weighted_signals"] = weighted

    return state

In [ ]:
def combine_signals(quant_score, news_score, risk):
    conflict = (quant_score * news_score < 0)

    if risk > 0.7:
        wq, wn = 0.8, 0.2
    elif abs(news_score) > 0.4:
        wq, wn = 0.5, 0.5
    else:
        wq, wn = 0.6, 0.4

    combined = wq * quant_score + wn * news_score
    if conflict:
        combined *= 0.7

    return combined, conflict

In [ ]:
from langgraph.graph import StateGraph

def build_graph(planner, news_agent, decision_agent):
    builder = StateGraph(State)

    builder.add_node("planner", planner)
    builder.add_node("market", market_agent)
    builder.add_node("technical", technical_agent)
    builder.add_node("options", options_agent)
    builder.add_node("news", news_agent)
    builder.add_node("risk", risk_agent)
    builder.add_node("decision", decision_agent)

    builder.set_entry_point("planner")

    builder.add_edge("planner", "market")
    builder.add_edge("market", "technical")
    builder.add_edge("technical", "options")
    builder.add_edge("options", "news")
    builder.add_edge("news", "risk")
    builder.add_edge("risk", "decision")

    builder.set_finish_point("decision")

    return builder.compile()

In [ ]:
graph = build_graph(planner, news_agent, decision_agent)

query = """
I hold an NVDA call option:
- Strike: 375
- Expiration: today (April 2)
- Position: long call

Should I:
- sell now,
- hold until expiration,
- or roll to a later expiration?

Consider:
- time decay (theta)
- technical indicators
- options signals
- news sentiment
"""

result = graph.invoke({
    "query": query,
    "ticker": "",
    "user_id": "default_user"
})

import pandas as pd

decision = result.get("decision", {})
risk_score = result.get("risk_score", "N/A")

df = pd.DataFrame({
    "Metric": ["Action", "Confidence", "Risk Score", "Combined Score", "Reason"],
    "Value": [
        decision.get("action", "N/A"),
        decision.get("confidence", "N/A"),
        risk_score,
        decision.get("combined_score", "N/A"),
        decision.get("reason", "N/A")
    ]
})

print("=== FINAL DECISION ===")
print(df)

print("\n=== LLM ANALYSIS ===")
print(result.get("llm_analysis", "No LLM analysis available"))

=== FINAL DECISION ===
           Metric                                              Value
0          Action                                               SELL
1      Confidence                                               0.85
2      Risk Score                                               0.41
3  Combined Score                                                N/A
4          Reason  Given that your NVDA call option with a $375 s...

=== LLM ANALYSIS ===
{
  "action": "SELL",
  "reason": "Given that your NVDA call option with a $375 strike expires today, time decay (theta) is at its maximum, which heavily erodes the option's value as expiration approaches. The RSI near 47 indicates a neutral technical outlook, offering no strong momentum support. The composite quant score is positive (18.0 with high confidence), suggesting some upside potential, but the near-term news sentiment is slightly negative (-0.051) and mixed overall, which may limit immediate gains. Additionally, your moderate

In [ ]:
def single_llm_decision(query):
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": query}]
    )

    return response.choices[0].message.content

In [ ]:
print("\n=== BASELINE (Single LLM) ===")
print(baseline)

print("\n=== MULTI-AGENT ===")
print(result.get("decision"))

print("\n=== RISK SCORE ===")
print(result.get("risk_score"))


=== BASELINE (Single LLM) ===
{'action': 'sell now', 'confidence': 0.85, 'reason': 'With the option expiring today, time decay (theta) is at its maximum, rapidly eroding any remaining time value. If the current market price of NVDA is near or below the strike of 375, the long call is at risk of expiring worthless. Technical indicators show mixed momentum with slight resistance near 375, and there are no strong bullish signals in the latest options flow or news sentiment to justify holding or rolling. Selling now preserves remaining value and avoids potential total loss at expiration.'}

=== MULTI-AGENT ===
{'action': 'SELL', 'reason': "Given that your NVDA call option with a $375 strike expires today, time decay (theta) is at its maximum, which heavily erodes the option's value as expiration approaches. The RSI near 47 indicates a neutral technical outlook, offering no strong momentum support. The composite quant score is positive (18.0 with high confidence), suggesting some upside po